# ライブドアニュースコーパスの分類をTF-IDFで実現する

In [1]:
from datasets import load_dataset

ds = load_dataset("mteb/LivedoorNewsClustering")

In [2]:
category_names = [
    'dokujo-tsushin',
    'it-life-hack',
    'kaden-channel',
    'livedoor-homme',
    'movie-enter',
    'peachy',
    'smax',
    'sports-watch',
    'topic-news',
]

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words='english', min_df=5)

In [4]:
train_embeddings = vectorizer.fit_transform(ds["train"]["sentences"])
val_embeddings = vectorizer.transform(ds["validation"]["sentences"])
test_embeddings = vectorizer.transform(ds["test"]["sentences"])

In [5]:
print(train_embeddings.shape)
print(val_embeddings.shape)
print(test_embeddings.shape)

(5163, 6306)
(1106, 6306)
(1107, 6306)


## SVMによる分類

* 訓練セットと検証をセットを結合する。
  * 後で交差検証をするため。

In [6]:
import numpy as np

embeddings = np.concatenate([train_embeddings.toarray(), val_embeddings.toarray()])
labels = np.array(list(ds["train"]["labels"]) + list(ds["validation"]["labels"]))

* 交差検証によりハイパーパラメータをチューニングする。

In [7]:
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import LinearSVC

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=1234)

best_score = 0
best_C = None
for C in 10. ** np.arange(-1, 4):
  print(f"C={C:.2e}")
  scores = []
  skf_split = skf.split(embeddings, labels)
  print("\tsplit-wise accuracies:", end="")
  for train_index, val_index in skf_split:
    X_train = embeddings[train_index]
    clf = LinearSVC(C=C, dual=False, max_iter=1000, random_state=123)
    clf.fit(X_train, labels[train_index])
    X_val = embeddings[val_index]
    score = clf.score(X_val, labels[val_index])
    print(f"\t{score:.3f}", end="")
    scores.append(score)
  mean_score = np.array(scores).mean()
  print(f"\n\tmean accuracy: {mean_score:.3f}")
  if mean_score > best_score:
    best_score = mean_score
    best_C = C
print(f"Best C: {best_C}, Best score: {best_score:.3f}")

C=1.00e-01
	split-wise accuracies:	0.886	0.901	0.892	0.883	0.896
	mean accuracy: 0.892
C=1.00e+00
	split-wise accuracies:	0.888	0.898	0.892	0.886	0.906
	mean accuracy: 0.894
C=1.00e+01
	split-wise accuracies:	0.873	0.887	0.876	0.872	0.887
	mean accuracy: 0.879
C=1.00e+02
	split-wise accuracies:	0.867	0.880	0.873	0.865	0.881
	mean accuracy: 0.873
C=1.00e+03
	split-wise accuracies:	0.864	0.876	0.874	0.868	0.879
	mean accuracy: 0.872
Best C: 1.0, Best score: 0.894


* 選ばれた設定をテストセットで評価する。

In [8]:
clf = LinearSVC(C=best_C, dual=False, max_iter=1000, random_state=123)
clf.fit(embeddings, labels)
score = clf.score(test_embeddings.toarray(), ds["test"]["labels"])
print(f"{score:.3f}")

0.893
